In [13]:
import numpy as np

In [14]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left          
        self.right = right        
        self.value = value 

def gini(y):
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    return 1 - np.sum(probabilities ** 2)

def mse(y):
    return np.mean((y - np.mean(y)) ** 2)

def stop_recursion_clf(y):
    values, counts = np.unique(y, return_counts=True)
    return Node(value=values[np.argmax(counts)])

def stop_recursion_reg(y):
    return Node(value=np.mean(y))

stop_functions = {
    "classification": lambda y: stop_recursion_clf(y),
    "regression": lambda y: stop_recursion_reg(y)
}

class DecisionTree:
    def __init__(self, task = 'classification', max_depth=None, min_samples_split=2):
        self.root = None
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        
        self.stop_function = {
            "classification": lambda y: stop_recursion_clf(y),
            "regression": lambda y: stop_recursion_reg(y)
        }[task]

        self.gain = {
            "classification": lambda y, left_idx, right_idx:  gini(y) - (len(y[left_idx])/len(y) * gini(y[left_idx]) + len(y[right_idx])/len(y) * gini(y[right_idx])),
            "regression": lambda y, left_idx, right_idx: mse(y) - (len(y[left_idx])/len(y) * mse(y[left_idx]) + len(y[right_idx])/len(y) * mse(y[right_idx]))
        }[task]

        self.stop_no_split = {
            "classification": lambda y: y[0],
            "regression": lambda y: np.mean(y)
        }[task]
    
    def _best_split(self, X, y):
        best_gain = -np.inf
        best_feature, best_threshold = None, None

        for feature in range(X.shape[1]):
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds:
                left_idx = X[:, feature] <= threshold
                right_idx = ~left_idx
                
                if len(y[left_idx]) == 0 or len(y[right_idx]) == 0:
                    continue

                gain = self.gain(y, left_idx, right_idx)

                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold

    def _build_tree(self, X, y, depth=0):

        if (self.max_depth and depth >= self.max_depth) or len(y) < self.min_samples_split:
            return self.stop_function(y)

        feature, threshold = self._best_split(X, y)

        if feature is None:
            return Node(value=self.stop_no_split(y))

        left_idx = X[:, feature] <= threshold
        right_idx = ~left_idx

        left = self._build_tree(X[left_idx], y[left_idx], depth+1)
        right = self._build_tree(X[right_idx], y[right_idx], depth+1)

        return Node(feature, threshold, left, right)
    
    def fit(self, X, y):
        self.root = self._build_tree(X, y)

    def _predict_sample(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature] <= node.threshold:
            return self._predict_sample(x, node.left)
        else:
            return self._predict_sample(x, node.right)
    
    def predict(self, X):
        return np.array([self._predict_sample(x, self.root) for x in X])

In [15]:
# from sklearn.datasets import load_iris
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score

# data = load_iris()
# X, y = data.data, data.target
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# tree = DecisionTree(task='classification', max_depth=3)
# tree.fit(X_train, y_train)
# preds = tree.predict(X_test)
# print(f"Accuracy: {accuracy_score(y_test, preds)}")

# X = np.sort(5 * np.random.rand(100, 1), axis=0)
# y = np.sin(X).ravel() + np.random.normal(0, 0.1, X.shape[0])

# tree = DecisionTree(task='regression', max_depth=3)
# tree.fit(X, y)
# preds = tree.predict(X)
# print(f"MSE: {np.mean((y - preds) ** 2)}")

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

In [17]:
data = pd.read_csv('../data/processed_smoke_detector.csv')
X = data.drop(['Fire Alarm'], axis=1)
y = data['Fire Alarm']
print(X.shape, y.shape)

(41247, 12) (41247,)


In [18]:
dtc = DecisionTree(task='classification', max_depth=5)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

In [20]:
dtc.fit(X_train.values, y_train.values)

In [21]:
y_pred = dtc.predict(X_test.values)

In [22]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[1808    2]
 [  11 6429]]
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      1810
           1       1.00      1.00      1.00      6440

    accuracy                           1.00      8250
   macro avg       1.00      1.00      1.00      8250
weighted avg       1.00      1.00      1.00      8250

